# Solar activity and revolutions: computable methods and results

[Open the project on GitHub](https://github.com/CastaliaInstitute/apocalypso)  
Public page: [apocalypso.castalia.institute/solar-revolutions](https://apocalypso.castalia.institute/solar-revolutions/)

## Plain-language bottom line

This notebook is the computable companion to the public page. It follows the same scientific structure: hypotheses, methods, statistical results, discussion, and conclusions.

The current evidence is best described as **exploratory / non-confirmatory**. The visual overlay can look suggestive, and one narrow rule has high specificity: exact solar-maximum years usually do **not** occur in quiet revolution years. But the stronger tests do not show that solar activity reliably predicts revolutions.

The important distinction is this:

- **Descriptive pattern:** Some solar-cycle peaks line up with high-revolution years.
- **Statistical result:** The high-specificity rule reproduces, but it has low sensitivity and weak balanced accuracy.
- **Predictive evidence:** Solar exposure does not improve held-out country-year onset prediction.
- **Causal evidence:** Country-specific exposure models with fixed effects and conventional controls do not confirm a solar mechanism.

<details>
<summary>What this notebook is for</summary>

The public page is written for readers. This notebook keeps the methods computable. It loads the same public JSON artifacts used by the page, recomputes the headline specificity endpoint from RED and SILSO data, and provides optional cells for rerunning the repository scripts that generate the deeper audit files.

</details>


## 1. Hypotheses

This section defines the hypothesis before looking at the results.

**H0: no solar influence**  
After accounting for time structure, model choice, controls, and geography, RED revolutionary activity is not meaningfully associated with SILSO solar activity beyond chance and confounding.

**H1: solar influence**  
Solar activity changes local radiation, geomagnetic, or auroral-proximity exposure; that exposure amplifies agricultural, conflict, communications, behavioral, or institutional stress; and vulnerable countries then experience higher new-revolution onset risk after fixed effects and conventional controls.

**Decision rule**  
H1 requires statistically meaningful onset effects, time-structured null rejection, local exposure identification, improved out-of-sample prediction, and robustness to political-economy, conflict, food/agriculture, geography, timing, and subtype checks. A visual annual overlay is not enough.


## 2. Setup

Run this section first. In Colab, the notebook will clone the public repository if it is not already present. Locally, it will search upward from the current directory.


In [ ]:
#@title Install notebook dependencies { display-mode: "form" }
INSTALL_PACKAGES = False  # Set True in a fresh Colab runtime if imports fail.

if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "pandas>=2.2,<3", "numpy>=1.26,<3", "matplotlib>=3.8,<4", "scipy>=1.11,<2"
    ], check=True)


In [ ]:
#@title Locate or clone the repository { display-mode: "form" }
import json
import math
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/CastaliaInstitute/apocalypso.git"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_repo_root() -> Path:
    env_root = os.environ.get("APOCALYPSO_REPO_ROOT") or os.environ.get("SOLAR_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/apocalypso")])
    for candidate in candidates:
        if (candidate / "dashboard" / "public" / "api" / "red").exists():
            return candidate
        if (candidate / "api" / "red").exists():
            return candidate
    if running_in_colab():
        target = Path("/content/apocalypso")
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        return target
    raise FileNotFoundError("Could not find repo root. Set APOCALYPSO_REPO_ROOT to the apocalypso checkout.")

ROOT = find_repo_root()
API = ROOT / "api" if (ROOT / "api" / "red").exists() else ROOT / "dashboard" / "public" / "api"
RED_API = API / "red"
print("Repo root:", ROOT)
print("RED API artifacts:", RED_API)


## 3. Load the public artifacts

These JSON files are the same data products used by the public `solar-revolutions` page. They are deliberately machine-readable so the page and notebook can agree.


In [ ]:
#@title Load RED, SILSO, and audit artifacts { display-mode: "form" }
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

annual_summary = load_json(RED_API / "annual-summary.json")
solar_monthly = load_json(API / "input-data" / "climate_solar_activity.json")
statistical_audit = load_json(RED_API / "statistical-audit.json")
primary_endpoint = load_json(RED_API / "primary-specificity-endpoint.json")
country_panel = load_json(RED_API / "country-panel-solar-model.json")
comparative_evidence = load_json(RED_API / "science-review-comparative-evidence.json")
monthly_timing = load_json(RED_API / "monthly-timing-diagnostic.json")
shape_synchrony = load_json(RED_API / "solar-shape-synchrony.json")

print("Audit verdict:")
print(statistical_audit["current_verdict"])


## 4. Methods and audit trail

This section is not a statistical result. It documents what data products and checks are available so the analysis can be repeated.

The public page now treats audit/completeness material as part of **Methods**, not **Results**. Results below are separated into two scopes: paper-scoped replication statistics using RED/SILSO, and follow-on research statistics using expanded geographic, political-economic, and control inputs.

<details>
<summary>Repeatable artifacts used here</summary>

- `dashboard/public/api/red/annual-summary.json`
- `dashboard/public/api/input-data/climate_solar_activity.json`
- `dashboard/public/api/red/primary-specificity-endpoint.json`
- `dashboard/public/api/red/country-panel-solar-model.json`
- `dashboard/public/api/red/science-review-comparative-evidence.json`
- `dashboard/public/api/red/monthly-timing-diagnostic.json`
- `dashboard/public/api/red/solar-shape-synchrony.json`
- `dashboard/public/api/red/statistical-audit.json`

</details>


In [ ]:
#@title Methods audit trail, not a statistical result { display-mode: "form" }
method_rows = [
    ["Scientific evidence classification", statistical_audit.get("scientific_evidence_grade")],
    ["Review/audit status", statistical_audit.get("review_quality_grade")],
    ["Documented checks", statistical_audit.get("review_readiness_counts", {}).get("documented")],
    ["Checks needing attention", statistical_audit.get("review_readiness_counts", {}).get("needs_attention")],
    ["Missing checks", statistical_audit.get("review_readiness_counts", {}).get("missing")],
]
pd.DataFrame(method_rows, columns=["methods item", "value"])


## 5. Paper-scoped statistics: Hernandez-paper replication inputs

This block stays close to the paper's observable ingredients: annual revolutionary activity from RED 1.0 and solar activity from SILSO. It asks whether the paper-style relationship can be reproduced and stress-tested using the same core input set.

### 5.1 Headline specificity endpoint

Plain-language version: we ask whether exact solar-maximum years behave like a warning light for high-revolution years. The answer is mixed. The light rarely turns on during quiet years, which gives high specificity. But it also misses most high-revolution years, so it is not a good general predictor.

<details>
<summary>Technical definition</summary>

- RED outcome: annual episode count.
- High-revolution year: year at or above the 75th percentile of annual RED episode counts.
- Solar signal: annualized SILSO sunspot number, smoothed with a 3-year centered moving average.
- Solar-positive year: exact local maximum in the smoothed annual SILSO series.
- Reported statistics: confusion matrix, sensitivity, specificity, positive predictive value, negative predictive value, balanced accuracy, and Matthews correlation.

</details>


In [ ]:
#@title Recompute annual RED and SILSO alignment { display-mode: "form" }
def parse_year(date_text):
    try:
        return int(str(date_text)[:4])
    except Exception:
        return None

red_rows = []
for point in annual_summary["points"]:
    year = point.get("year") or parse_year(point.get("date"))
    if year is None:
        continue
    if 1900 <= int(year) <= 2014:
        red_rows.append({
            "year": int(year),
            "episodes": float(point.get("episodes") or 0),
            "participants_proxy": float(point.get("participants_proxy") or 0),
        })
red = pd.DataFrame(red_rows).drop_duplicates("year").sort_values("year")

solar_rows = []
for point in solar_monthly["points"]:
    year = parse_year(point.get("date"))
    value = pd.to_numeric(point.get("value"), errors="coerce")
    if year is not None and 1900 <= year <= 2014 and pd.notna(value):
        solar_rows.append({"year": year, "solar": float(value)})
solar = pd.DataFrame(solar_rows).groupby("year", as_index=False)["solar"].mean()

panel = pd.DataFrame({"year": list(range(1900, 2015))})
panel = panel.merge(red, on="year", how="left").merge(solar, on="year", how="left")
panel["episodes"] = panel["episodes"].fillna(0.0)
panel["participants_proxy"] = panel["participants_proxy"].fillna(0.0)
panel.head()


In [ ]:
#@title Compute solar maxima and classification statistics { display-mode: "form" }
def moving_average(values, window=3):
    values = list(values)
    radius = window // 2
    out = []
    for i in range(len(values)):
        lo = max(0, i - radius)
        hi = min(len(values), i + radius + 1)
        out.append(float(np.mean(values[lo:hi])))
    return out

def local_maxima(values):
    peaks = []
    for i in range(1, len(values) - 1):
        if values[i] >= values[i - 1] and values[i] >= values[i + 1] and (values[i] > values[i - 1] or values[i] > values[i + 1]):
            peaks.append(i)
    return peaks

def binary_stats(predicted, actual):
    predicted = np.asarray(predicted, dtype=bool)
    actual = np.asarray(actual, dtype=bool)
    tp = int(np.sum(predicted & actual))
    fp = int(np.sum(predicted & ~actual))
    fn = int(np.sum(~predicted & actual))
    tn = int(np.sum(~predicted & ~actual))
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else np.nan
    return pd.Series({
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "positive_predictive_value": ppv,
        "negative_predictive_value": npv,
        "balanced_accuracy": balanced_accuracy,
        "matthews_correlation": mcc,
    })

panel["solar_smoothed_3y"] = moving_average(panel["solar"], 3)
peak_indexes = set(local_maxima(panel["solar_smoothed_3y"].tolist()))
panel["solar_peak_exact"] = [i in peak_indexes for i in range(len(panel))]
threshold = panel["episodes"].quantile(0.75)
panel["high_revolution_year"] = panel["episodes"] >= threshold

headline_stats = binary_stats(panel["solar_peak_exact"], panel["high_revolution_year"])
print("High-revolution threshold:", threshold)
headline_stats.to_frame("recomputed")


In [ ]:
#@title Plot the visual overlay that motivated the stronger tests { display-mode: "form" }
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(panel["year"], panel["episodes"], color="#9b7a2a", alpha=0.55, label="RED episodes")
ax1.set_ylabel("RED annual episode count")
ax1.set_xlabel("Year")

ax2 = ax1.twinx()
ax2.plot(panel["year"], panel["solar_smoothed_3y"], color="#7fc7ff", linewidth=2.2, label="SILSO, 3-year smoothed")
ax2.scatter(panel.loc[panel["solar_peak_exact"], "year"], panel.loc[panel["solar_peak_exact"], "solar_smoothed_3y"], color="white", edgecolor="#111", zorder=5, label="Exact solar maximum")
ax2.set_ylabel("Sunspot number")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.set_title("The overlay can look suggestive, but visual similarity is not enough")
plt.tight_layout()
plt.show()


### 5.2 Paper-scoped statistical summary for nontechnical readers

The current statistical results support a cautious interpretation.

- The headline specificity statistic is real and reproducible: exact solar-maximum years reject most quiet revolution years.
- The same rule has low sensitivity: it misses most high-revolution years.
- Balanced accuracy is only slightly above chance for the headline endpoint.
- The visual overlay is not strong enough under circular-shift and shape-synchrony checks.
- Country-year models with local exposure do not confirm the solar hypothesis.
- Held-out prediction gets worse when solar exposure is added.
- Conventional political-economic and mobilization variables often describe the revolution data better than SILSO.

<details>
<summary>Exact reported statistics from the current artifacts</summary>

| Test | Current result | Interpretation |
|---|---:|---|
| Specificity of exact solar-maximum rule | 93.6% | High specificity, low false-alarm rate |
| Sensitivity of exact solar-maximum rule | 13.5% | Misses most high-revolution years |
| Balanced accuracy | 53.6% | Only slightly above chance under this endpoint |
| Country+year fixed-effect geomagnetic exposure p-value | 0.761 | No confirmatory local-exposure effect |
| Overall held-out onset delta AUC after adding solar | -0.010 | Prediction worsens |
| Overall held-out onset delta log loss after adding solar | +0.529 | Prediction worsens |
| Vulnerable-country shifted-exposure placebo p-value | 0.528 | Does not beat placebo timing |

</details>


In [ ]:
#@title Pull the exact headline numbers from generated artifacts { display-mode: "form" }
observed = primary_endpoint["observed"]
country_model = country_panel["models"]["event_country_year_fe_solar_geomagnetic_exposure"]

summary_rows = [
    ["Specificity", observed["specificity"]],
    ["Sensitivity", observed["sensitivity"]],
    ["Balanced accuracy", observed["balanced_accuracy"]],
    ["Matthews correlation", observed["matthews_correlation"]],
    ["Country+year FE exposure p", country_model["solar_exposure_p_clustered_by_year"]],
]

pd.DataFrame(summary_rows, columns=["quantity", "value"])


### 5.3 Paper-scoped follow-on statistics

These tests still use only the Hernandez-paper input scope: RED revolutionary activity and SILSO solar activity. They do **not** introduce country controls, macroeconomic predictors, NAVCO, V-Dem, WDI, or NASA POWER. The purpose is to ask whether the paper-style RED/SILSO pattern survives reasonable statistical choices.

Included here:

- same-year Pearson and Spearman correlation
- annual lag correlations
- quantile/window sensitivity for the high-revolution classifier
- split-sample threshold checks
- leave-one-solar-cycle-out influence checks


In [ ]:
#@title Paper-scoped correlations and lag scan: RED/SILSO only { display-mode: "form" }
from scipy.stats import spearmanr

x = panel["solar"].astype(float).to_numpy()
y = panel["episodes"].astype(float).to_numpy()
pearson_same_year = float(np.corrcoef(x, y)[0, 1])
spearman_same_year = float(spearmanr(x, y, nan_policy="omit").statistic)

lag_rows = []
for lag in range(-8, 9):
    if lag < 0:
        xs = x[-lag:]
        ys = y[:len(y) + lag]
    elif lag > 0:
        xs = x[:-lag]
        ys = y[lag:]
    else:
        xs = x
        ys = y
    lag_rows.append({
        "lag_years": lag,
        "interpretation": "solar leads RED" if lag > 0 else "RED leads solar" if lag < 0 else "same year",
        "n": len(xs),
        "pearson_r": float(np.corrcoef(xs, ys)[0, 1]),
    })

print("Same-year Pearson r:", round(pearson_same_year, 4))
print("Same-year Spearman rho:", round(spearman_same_year, 4))
pd.DataFrame(lag_rows).sort_values("pearson_r", key=lambda s: s.abs(), ascending=False).head(10)


In [ ]:
#@title Paper-scoped threshold/window sensitivity: RED/SILSO only { display-mode: "form" }
grid = pd.DataFrame(primary_endpoint["quantization_grid"]["rows"])
flat_rows = []
for row in primary_endpoint["quantization_grid"]["rows"]:
    obs = row["observed"]
    flat_rows.append({
        "threshold_rule": row["rule"],
        "quantile": row["quantile"],
        "episode_threshold": row["threshold"],
        "solar_window_radius_years": row["solar_window_radius"],
        "specificity": obs.get("specificity"),
        "sensitivity": obs.get("sensitivity"),
        "balanced_accuracy": obs.get("balanced_accuracy"),
        "matthews_correlation": obs.get("matthews_correlation"),
        "TP": obs.get("TP"),
        "FP": obs.get("FP"),
        "FN": obs.get("FN"),
        "TN": obs.get("TN"),
    })
flat = pd.DataFrame(flat_rows)
print("Rows in predeclared quantile/window grid:", len(flat))
print("Best balanced accuracy in grid:", round(float(flat["balanced_accuracy"].max()), 4))
flat.sort_values(["balanced_accuracy", "specificity"], ascending=False).head(12)


In [ ]:
#@title Paper-scoped split-sample and leave-one-cycle sensitivity { display-mode: "form" }
rows = []
selected = primary_endpoint.get("split_validation", {}).get("selected_rule")
fixed = primary_endpoint.get("split_validation", {}).get("fixed_primary_rule")
if selected:
    rows.append({
        "check": "selected on first half, tested on second half",
        "specificity": selected["test"].get("specificity"),
        "sensitivity": selected["test"].get("sensitivity"),
        "balanced_accuracy": selected["test"].get("balanced_accuracy"),
    })
if fixed:
    rows.append({
        "check": "fixed primary rule, tested on second half",
        "specificity": fixed["test"].get("specificity"),
        "sensitivity": fixed["test"].get("sensitivity"),
        "balanced_accuracy": fixed["test"].get("balanced_accuracy"),
    })

split_df = pd.DataFrame(rows)
cycle_rows = []
for row in primary_endpoint.get("cycle_influence", {}).get("rows", []):
    obs = row["observed"]
    cycle_rows.append({
        "omitted_cycle": f"{row['omitted_start']}-{row['omitted_end']}",
        "omitted_peak_year": row["omitted_peak_year"],
        "specificity": obs.get("specificity"),
        "sensitivity": obs.get("sensitivity"),
        "balanced_accuracy": obs.get("balanced_accuracy"),
        "TP": obs.get("TP"), "FP": obs.get("FP"), "FN": obs.get("FN"), "TN": obs.get("TN"),
    })

print("Split-sample checks")
display(split_df)
print("Leave-one-solar-cycle-out checks")
pd.DataFrame(cycle_rows).head(20)


### 5.4 Shape-synchrony tests

This section asks whether the apparent visual similarity can be captured numerically. Current shape tests remain weak: smoothed co-movement is small, and the circular-shift test does not show that the real solar sequence is clearly better than shifted versions.


In [ ]:
#@title Shape-synchrony diagnostics { display-mode: "form" }
shape_summary = shape_synchrony["smoothed_comovement"]
print("Smoothed Pearson r:", shape_summary["pearson"])
print("Smoothed Spearman r:", shape_summary["spearman"])
print("Circular-shift p-value:", shape_summary["circular_shift_p_two_sided"])

pd.DataFrame(shape_synchrony["lag_correlations"]).head(20)


## 6. Follow-on research statistics: expanded inputs

This block goes beyond the paper. It asks whether the solar hypothesis survives stronger tests using country-year exposure, fixed effects, NASA POWER radiation, V-Dem, WDI, NAVCO, food/agriculture, conflict, macro-financial, and other historical controls. These are follow-on diagnostics, not direct replication of the paper's original input set.

### 6.1 Country fixed effects, explained simply

A country fixed effect means the model compares a country mostly with itself over time, rather than comparing very different countries to each other. This matters because countries differ in stable ways: geography, institutions, culture, colonial history, latitude, and many other factors. Fixed effects remove those stable differences from the estimate.

A year fixed effect means the model also removes shocks shared by the whole world in a given year, such as world wars, global depressions, or global waves of decolonization.

Why this matters here: global SILSO sunspots vary only by year. If we include year fixed effects, a purely global solar number is absorbed by the year effect. To test solar activity with both country and year fixed effects, we need **country-specific exposure**, such as solar activity multiplied by latitude, geomagnetic sensitivity, surface radiation, or auroral proximity.

<details>
<summary>Technical note</summary>

The country-panel artifact includes both weaker country-fixed-effect/time-trend diagnostics and stronger two-way fixed-effect exposure diagnostics. The two-way models use country-varying solar-geomagnetic exposure so the solar term is not perfectly collinear with year fixed effects.

</details>


In [ ]:
#@title Inspect country-panel model results { display-mode: "form" }
model_rows = []
for key, model in country_panel.get("models", {}).items():
    row = {"model": key, "n": model.get("n"), "countries": model.get("countries")}
    for stat_key in [
        "solar_beta_per_1sd",
        "solar_p_approx",
        "solar_p_clustered_by_year",
        "solar_exposure_beta",
        "solar_exposure_p_clustered_by_year",
        "within_r_squared",
        "r_squared",
    ]:
        if stat_key in model:
            row[stat_key] = model[stat_key]
    model_rows.append(row)

pd.DataFrame(model_rows)


### 6.2 Compare solar activity with alternative explanations

A stronger scientific case would require solar activity to add information beyond obvious historical drivers: economic stress, food prices, conflict, regime type, civil society, protest activity, and diffusion from nearby revolutions.

In the current descriptive scan, many non-solar inputs are more associated with RED annual revolutionary activity than SILSO is. That does not prove those alternatives are causal, but it makes the solar hypothesis harder to defend unless solar improves prediction after those alternatives are included.

<details>
<summary>Technical note</summary>

This is a descriptive global annual screen, not a causal model. It is useful as a triage test: if solar is weaker than many more plausible controls, the solar hypothesis should not be elevated without stronger country-year or event-history evidence.

</details>


In [ ]:
#@title Show the top non-solar correlates of RED annual episodes { display-mode: "form" }
red_outcome = next(o for o in comparative_evidence["outcomes"] if o["id"] == "red_episodes")
controls = pd.DataFrame(red_outcome["non_solar_controls_ranked"])
print("SILSO correlation with RED episodes:", red_outcome["solar_univariate"]["r"])
print("Non-solar controls stronger than SILSO:", sum(controls["abs_r"] > abs(red_outcome["solar_univariate"]["r"])))
controls[["id", "name", "source", "n", "r", "abs_r"]].head(15)


### 6.3 Timing tests

If solar activity has a short-term behavioral or institutional effect, monthly timing should help. The current monthly tests do not show strong timing evidence: same-month and lagged SILSO correlations are small and do not stand out against circular-shift nulls.

<details>
<summary>Technical note</summary>

Circular-shift tests keep the internal structure of the solar cycle but rotate it in time. This is a stronger placebo than treating each year/month as independent because solar activity is autocorrelated.

</details>


In [ ]:
#@title Monthly timing diagnostics { display-mode: "form" }
monthly_rows = []
for row in monthly_timing["monthly_lag_correlations"]:
    monthly_rows.append({
        "lag_months": row["lag_months"],
        "exposure": row["exposure"],
        "pearson_r": row["pearson_r"],
        "circular_shift_p_two_sided": row["circular_shift_null"]["p_two_sided"],
    })
pd.DataFrame(monthly_rows)


## 7. Optional: rerun the repository scripts

The page artifacts are generated by scripts in the repository. Running them can take longer than the lightweight notebook cells above, but this is the repeatable path for refreshing the public JSON outputs.

<details>
<summary>Scripts represented in this notebook</summary>

- `scripts/build_solar_revolutions_audit_report.py`
- `scripts/build_red_country_panel_solar_model.py`
- `scripts/build_solar_science_review_evidence.py`
- `scripts/build_red_monthly_timing_diagnostic.py`
- `scripts/build_solar_science_strengthening_matrix.py`

</details>


In [ ]:
#@title Optional full artifact regeneration { display-mode: "form" }
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    scripts = [
        "scripts/build_red_country_panel_solar_model.py",
        "scripts/build_solar_science_review_evidence.py",
        "scripts/build_red_monthly_timing_diagnostic.py",
        "scripts/build_solar_science_strengthening_matrix.py",
        "scripts/build_solar_revolutions_audit_report.py",
    ]
    for script in scripts:
        print("Running", script)
        subprocess.run([sys.executable, str(ROOT / script)], check=True)
    print("Pipeline complete. Reload artifacts above to inspect regenerated results.")
else:
    print("RUN_FULL_PIPELINE is False. Set it to True to regenerate artifacts.")


## 8. Discussion and conclusions

The solar-influence hypothesis is **not confirmed** by the current evidence.

The audited headline result is a narrow classification finding: exact smoothed SILSO solar-maximum years reject about 93.6% of non-high-revolution years under the top-quartile RED episode-count endpoint. That is useful as a low-false-alarm descriptive screen.

It is not, by itself, strong predictive evidence. The same endpoint has low sensitivity, balanced accuracy near 0.536, and weak support under random-window, circular-shift, and shape-synchrony checks.

The stronger country-year and event-history tests point the same way. Country-varying solar-geomagnetic exposure is near zero and non-significant in the two-way fixed-effect event model, and adding solar exposure worsens held-out onset prediction.

The defensible conclusion is therefore:

**The paper-style relationship is reproducible and visually/statistically browsable, but causal or operational-prediction language is not justified by the current evidence.**

The next scientifically stronger version would need preregistered country-month or country-year event-history models with local geomagnetic/radiation exposure, vulnerability interactions, neighboring-revolution diffusion, and out-of-sample validation against placebo solar cycles.
